<a href="https://colab.research.google.com/github/malikbaqi12/CLO-based-Course-Analysis/blob/main/Academic_Result_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Academic Result Analytics Platform
**IQRA University Chak Shahzad Campus | BS Artificial Intelligence**

---
**Supports:** Theory & Lab courses | Any CLO/GA count | Auto-detects pass mark, weightages, course type

**How to run:**
1. `Runtime → Run all`
2. Cell 2 prompts you to upload your result workbook
3. All charts appear inline; HTML dashboard and CSV download at the end


In [16]:
pip install matplotlib

In [17]:
pip install rich-gradient

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 28.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
bigframes 2.41.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
pyiceberg 

In [18]:
# ── Cell 1: Install ────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['openpyxl', 'plotly', 'kaleido', 'pandas', 'numpy']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ Libraries ready')

✅ Libraries ready


In [19]:
# ── Cell 2: Upload ─────────────────────────────────────────────────────────────
try:
    from google.colab import files as _gf
    print('📂 Upload your result workbook (.xlsx)')
    _up = _gf.upload()
    EXCEL_PATH = list(_up.keys())[0]
    print(f'✅ Uploaded: {EXCEL_PATH}')
    IN_COLAB = True
except ImportError:
    EXCEL_PATH = 'result_sheet.xlsx'   # ← set path for local run
    IN_COLAB = False
    print(f'ℹ️  Local mode → {EXCEL_PATH}')

📂 Upload your result workbook (.xlsx)


Saving OOP (BSAI) - Fall 2025 - Result Sheet.xlsx to OOP (BSAI) - Fall 2025 - Result Sheet (1).xlsx
✅ Uploaded: OOP (BSAI) - Fall 2025 - Result Sheet (1).xlsx


In [20]:
# ── Cell 3: Configuration (edit if needed) ─────────────────────────────────────
PASS_MARK_FALLBACK = 60   # used only if auto-detection fails
CLO_THRESHOLD      = 50   # CLO attainment threshold (%)
TOP_N              = 10   # top/bottom students shown

C = dict(
    primary='#1a73e8', success='#34a853', danger='#ea4335',
    warning='#fbbc04', info='#46bdc6', purple='#9c27b0', bg='#ffffff',
)
GRADE_ORDER = ['A','A-','B+','B','B-','C+','C','C-','D+','D','F']
GRADE_COLORS = {
    'A':'#1565c0','A-':'#1976d2',
    'B+':'#2e7d32','B':'#388e3c','B-':'#43a047',
    'C+':'#f57f17','C':'#f9a825','C-':'#fbc02d',
    'D+':'#bf360c','D':'#e65100','F':'#b71c1c',
}
def gc(g): return GRADE_COLORS.get(str(g).strip(), '#90a4ae')
print(f'⚙️  CLO threshold={CLO_THRESHOLD}% | Pass fallback={PASS_MARK_FALLBACK}%')

⚙️  CLO threshold=50% | Pass fallback=60%


In [21]:
# ── Cell 4: Load engine & parse ────────────────────────────────────────────────
import io, re, warnings
import numpy as np
import pandas as pd
import openpyxl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe_float(v, default=0.0):
    if v is None: return default
    if isinstance(v, str) and v.strip().startswith('#'): return default
    try:
        f = float(v)
        return default if (f != f) else f
    except: return default

def rts(v): return str(v).strip() if v is not None else ''

def col_of(hdr, *kws):
    for ci, c in enumerate(hdr):
        s = rts(c).lower()
        if any(k in s for k in kws): return ci
    return None

def find_sheet(wb, *kws):
    for name in wb.sheetnames:
        if any(k.lower() in name.lower() for k in kws): return name
    return None

# ── Open workbook ─────────────────────────────────────────────────────────────
with open(EXCEL_PATH, 'rb') as f:
    fb = f.read()
wb = openpyxl.load_workbook(io.BytesIO(fb), data_only=True)
print('Sheets:', wb.sheetnames)

SH_MAIN  = find_sheet(wb, 'final combined')
SH_CLO   = find_sheet(wb, 'quantized')
SH_WT    = find_sheet(wb, 'weighted theory', 'weighted marks')
print(f'Main: {SH_MAIN} | CLO: {SH_CLO} | Weights: {SH_WT}')
assert SH_MAIN and SH_CLO, 'Required sheets not found!'

ws_main = wb[SH_MAIN]
ws_clo  = wb[SH_CLO]

Sheets: ['Complete Data', 'Pre-Final-Exam Assessment', 'Mid & End Term Marks Sheet', 'Weighted Theory Marks Sheet ', 'Final Combined Marks Sheet', 'Award List (Theory)', 'Quantized Result', 'Percentages']
Main: Final Combined Marks Sheet | CLO: Quantized Result | Weights: Weighted Theory Marks Sheet 


In [22]:
# ── Cell 5: Metadata, pass mark, course type ──────────────────────────────────
_meta = {}
for row in ws_main.iter_rows(min_row=3, max_row=9, values_only=True):
    if row[0] and row[2]: _meta[rts(row[0]).lower()] = rts(row[2])
    elif row[0] and row[7]: _meta[rts(row[0]).lower()] = rts(row[7])

def gm(*keys, default='N/A'):
    for k in keys:
        for mk, mv in _meta.items():
            if k.lower() in mk: return mv
    return default

COURSE_CODE  = gm('course code')
COURSE_TITLE = gm('course title')
FACULTY      = gm('faculty','name of faculty','instructor')
SEMESTER     = gm('offered in','semester')
PROGRAM      = gm('program')
BATCH        = gm('batch')

# Detect course type
hdr10 = list(ws_main.iter_rows(min_row=10, max_row=10, values_only=True))[0]
hdr10_str = ' '.join(rts(h).lower() for h in hdr10 if h)
IS_LAB = any(k in hdr10_str for k in ['lab report','lab project','lab mid','viva','open ended'])

# Auto-detect pass mark
PASS_MARK = PASS_MARK_FALLBACK
for row in ws_main.iter_rows(min_row=3, max_row=12, values_only=True):
    for cell in row:
        m = re.search(r'less\s+than\s+(\d+)', rts(cell).lower())
        if m: PASS_MARK = int(m.group(1))+1; break

print(f'Course  : {COURSE_CODE} — {COURSE_TITLE}')
print(f'Faculty : {FACULTY}')
print(f'Program : {PROGRAM} | Batch: {BATCH} | Semester: {SEMESTER}')
print(f'Type    : {"LAB" if IS_LAB else "THEORY"}')
print(f'PassMark: {PASS_MARK}%')

Course  : CMC-112 — Object Oriented Programming
Faculty : Mr. Abdul Baqi Malik
Program : Artificial Intelligence | Batch: N/A | Semester: Fall 2025
Type    : THEORY
PassMark: 50%


In [23]:
# ── Cell 6: Extract student marks ─────────────────────────────────────────────
HDR_ROW  = 10
DATA_ROW = 12
_hdr = list(ws_main.iter_rows(min_row=HDR_ROW, max_row=HDR_ROW, values_only=True))[0]

CI = {'sr': col_of(_hdr,'sr','s.no'), 'reg': col_of(_hdr,'regn','reg','roll'),
      'name': col_of(_hdr,'name'), 'mid': col_of(_hdr,'mid term','lab mid'),
      'end': col_of(_hdr,'end term','lab end'), 'grade': col_of(_hdr,'grade'),
      'total': col_of(_hdr,'total marks','total weighted','theory weighted')}
if IS_LAB:
    CI.update({'lab_reports': col_of(_hdr,'lab report'), 'lab_project': col_of(_hdr,'lab project'),
               'viva': col_of(_hdr,'viva'), 'open_ended': col_of(_hdr,'open ended')})
else:
    CI.update({'assignment': col_of(_hdr,'assg','assignment'), 'quiz': col_of(_hdr,'quiz'),
               'presentation': col_of(_hdr,'presentation','project')})

records = []
for row in ws_main.iter_rows(min_row=DATA_ROW, max_row=1000, values_only=True):
    sr = row[CI['sr']] if CI.get('sr') is not None and CI['sr'] < len(row) else None
    if not isinstance(sr, (int,float)): continue
    reg  = rts(row[CI['reg']])  if CI.get('reg')  is not None else ''
    name = rts(row[CI['name']]) if CI.get('name') is not None else ''
    if not reg or reg.lower() in ('none','nan',''): continue
    def g(k, d=0.0):
        ci = CI.get(k)
        return safe_float(row[ci], d) if ci is not None and ci < len(row) else d
    total = g('total'); grade = rts(row[CI['grade']]) if CI.get('grade') is not None else ''
    rec = {'Sr':int(sr),'Reg_No':reg,'Name':name,'TotalMarks':round(total,2),
           'Percentage':round(total,2),'Grade':grade,
           'Pass_Fail':'Pass' if total>=PASS_MARK else 'Fail'}
    if IS_LAB:
        for k in ('lab_reports','lab_project','viva','open_ended','mid','end'):
            rec[k.title().replace('_','')] = g(k)
    else:
        for k in ('assignment','quiz','presentation','mid','end'):
            rec[k.title()] = g(k)
    records.append(rec)

df = pd.DataFrame(records)
print(f'✅ {len(df)} students extracted')
df.head(3)

✅ 23 students extracted


,Sr,Reg_No,Name,TotalMarks,Percentage,Grade,Pass_Fail,Assignment,Quiz,Presentation,Mid,End
0,1,052-25-126005,Naveed Ul Hassan,3.67,3.67,F,Fail,1.666667,0.0,0.0,2.0,0.0
1,2,052-25-126006,Muhammad Zaid,38.50,38.50,F,Fail,4.000000,4.5,13.0,17.0,0.0
2,3,052-25-126020,Muhammad Sudais,44.33,44.33,F,Fail,8.333333,7.0,13.0,16.0,0.0


In [24]:
# ── Cell 7: Extract CLO & GA data ─────────────────────────────────────────────
# Find CLO header row
CLO_HDR_ROW = None
for ri, row in enumerate(ws_clo.iter_rows(min_row=1, max_row=25, values_only=True), 1):
    if sum(1 for v in row if v and rts(v).upper().startswith('CLO ')) >= 2:
        CLO_HDR_ROW = ri; break

assert CLO_HDR_ROW, 'CLO header not found in Quantized sheet'
print(f'CLO header at row {CLO_HDR_ROW}')

_clo_hdr = list(ws_clo.iter_rows(min_row=CLO_HDR_ROW, max_row=CLO_HDR_ROW, values_only=True))[0]
CLO_COLS = [(ci,rts(v)) for ci,v in enumerate(_clo_hdr)
            if v and rts(v).upper().startswith('CLO ') and ci >= 3
            and rts(v) not in [c[1] for c in []]]
# deduplicate
seen = set(); CLO_COLS = [(ci,n) for ci,n in CLO_COLS if not (n in seen or seen.add(n))]
CLO_NAMES = [c[1] for c in CLO_COLS]
print(f'CLOs detected: {CLO_NAMES}')

CLO_DATA_START = CLO_HDR_ROW + 2
student_clo = {}   # reg → {clo: value}
for row in ws_clo.iter_rows(min_row=CLO_DATA_START, max_row=2000, values_only=True):
    if not row or not isinstance(row[0], (int,float)): continue
    reg = rts(row[1])
    if not reg or reg.lower() in ('none','nan',''): continue
    student_clo[reg] = {n: safe_float(row[ci]) for ci,n in CLO_COLS if ci<len(row)}

# Merge CLO values into df
for clo in CLO_NAMES:
    per_s = {reg: student_clo[reg].get(clo,0) for reg in student_clo}
    df[clo]           = df['Reg_No'].map(per_s).fillna(0)
    df[f'{clo}_Pass'] = df[clo].apply(lambda v:'Pass' if v>=CLO_THRESHOLD else 'Fail')

print(f'✅ CLO data merged into df')

CLO header at row 11
CLOs detected: ['CLO 1', 'CLO 2', 'CLO 3', 'CLO 4']
✅ CLO data merged into df


In [25]:
# ── Cell 8: Summary statistics ────────────────────────────────────────────────
N        = len(df)
PASSED   = int((df['Pass_Fail']=='Pass').sum())
FAILED   = N - PASSED
PASS_PCT = round(PASSED/N*100,1) if N else 0
FAIL_PCT = round(100-PASS_PCT,1)
AVG  = round(df['Percentage'].mean(),2)
HIGH = round(df['Percentage'].max(),2)
LOW  = round(df['Percentage'].min(),2)
MED  = round(df['Percentage'].median(),2)
STD  = round(df['Percentage'].std(),2)

_grades_present = [g for g in GRADE_ORDER if g in df['Grade'].values]
grade_dist = df['Grade'].value_counts().reindex(_grades_present).dropna().astype(int)

CLO_STATS = {}
for clo in CLO_NAMES:
    pc = int((df[f'{clo}_Pass']=='Pass').sum())
    CLO_STATS[clo] = {'avg':round(df[clo].mean(),1),'pass_cnt':pc,
                      'fail_cnt':N-pc,'pass_pct':round(pc/N*100,1) if N else 0}

print(f'Students={N} | Pass={PASSED}({PASS_PCT}%) | Fail={FAILED}({FAIL_PCT}%)')
print(f'Avg={AVG}% | High={HIGH}% | Low={LOW}% | Median={MED}% | Std={STD}%')
if CLO_STATS:
    print('\nCLO Summary:')
    for c,s in CLO_STATS.items():
        status = '✅' if s['avg']>=CLO_THRESHOLD else '❌'
        print(f'  {status} {c}: avg={s["avg"]}% | pass={s["pass_cnt"]}({s["pass_pct"]}%)')

Students=23 | Pass=12(52.2%) | Fail=11(47.8%)
Avg=48.55% | High=90.0% | Low=0.0% | Median=56.5% | Std=30.09%

CLO Summary:
  ❌ CLO 1: avg=43.3% | pass=10(43.5%)
  ❌ CLO 2: avg=46.8% | pass=11(47.8%)
  ✅ CLO 3: avg=50.5% | pass=14(60.9%)
  ✅ CLO 4: avg=52.9% | pass=13(56.5%)


In [26]:
# ── Cell 9: KPI Banner ────────────────────────────────────────────────────────
display(HTML(f'''
<div style="background:linear-gradient(135deg,#1a237e,#1565c0);color:#fff;
     border-radius:12px;padding:16px 24px;margin-bottom:16px;font-family:Segoe UI">
  <h2 style="margin:0 0 4px;font-size:20px">🎓 {COURSE_CODE} — {COURSE_TITLE}</h2>
  <p style="margin:0;font-size:12px;opacity:.85">Faculty: {FACULTY} | {PROGRAM} | {BATCH} | {SEMESTER} | {"LAB" if IS_LAB else "THEORY"}</p>
</div>
<div style="display:flex;flex-wrap:wrap;gap:10px;font-family:Segoe UI;margin-bottom:8px">
  {" ".join(f"""<div style="flex:1 1 110px;border-radius:10px;padding:14px;color:#fff;text-align:center;background:{bg};box-shadow:0 3px 10px rgba(0,0,0,.18)">
    <div style="font-size:9px;font-weight:700;opacity:.82;letter-spacing:.8px">{lbl}</div>
    <div style="font-size:26px;font-weight:800;margin:5px 0 2px">{val}</div>
    <div style="font-size:11px;opacity:.78">{sub}</div></div>"""
  for lbl,val,sub,bg in [
    ('Students',N,'enrolled','linear-gradient(135deg,#1a73e8,#0d47a1)'),
    ('Passed',PASSED,f'{PASS_PCT}% rate','linear-gradient(135deg,#34a853,#1b5e20)'),
    ('Failed',FAILED,f'{FAIL_PCT}% rate','linear-gradient(135deg,#ea4335,#b71c1c)'),
    ('Average',f'{AVG}%','class mean','linear-gradient(135deg,#f9a825,#e65100)'),
    ('Highest',f'{HIGH}%','top score','linear-gradient(135deg,#2e7d32,#1b5e20)'),
    ('Lowest',f'{LOW}%','bottom score','linear-gradient(135deg,#c62828,#7f0000)'),
    ('Std Dev',f'{STD}%','spread','linear-gradient(135deg,#9c27b0,#4a148c)'),
  ])}
</div>'''))

In [27]:
# ── Cell 10: Chart 1 – Pass/Fail & Grade Pies ─────────────────────────────────
fig1 = make_subplots(1,2,specs=[[{'type':'pie'},{'type':'pie'}]],
                     subplot_titles=['Pass vs Fail','Grade Distribution'])
fig1.add_trace(go.Pie(labels=['Pass','Fail'],values=[PASSED,FAILED],
    marker_colors=[C['success'],C['danger']],textinfo='label+percent+value',hole=0.42),1,1)
fig1.add_trace(go.Pie(labels=list(grade_dist.index),values=list(grade_dist.values),
    marker_colors=[gc(g) for g in grade_dist.index],textinfo='label+value',hole=0.42),1,2)
fig1.update_layout(height=430,title=f'<b>Pass/Fail & Grade — {COURSE_CODE}</b>',
                   legend=dict(orientation='h',y=-0.1))
fig1.show()

In [28]:
# ── Cell 11: Chart 2 – Grade Bar + Marks Histogram ───────────────────────────
fig2 = go.Figure(go.Bar(x=list(grade_dist.index),y=list(grade_dist.values),
    marker_color=[gc(g) for g in grade_dist.index],
    text=list(grade_dist.values),textposition='outside'))
fig2.update_layout(title='<b>Grade Distribution</b>',height=420,
    xaxis=dict(categoryorder='array',categoryarray=_grades_present),
    yaxis=dict(gridcolor='#eee'),plot_bgcolor='#f9f9f9')
fig2.show()

fig3 = go.Figure(go.Histogram(x=df['Percentage'],nbinsx=16,
                               marker_color=C['primary'],opacity=0.85))
fig3.add_vline(x=AVG,line_dash='dash',line_color=C['warning'],annotation_text=f'Avg {AVG}%')
fig3.add_vline(x=PASS_MARK,line_dash='dot',line_color=C['danger'],annotation_text=f'Pass {PASS_MARK}%')
fig3.add_vrect(x0=PASS_MARK,x1=100,fillcolor=C['success'],opacity=0.06,layer='below',line_width=0)
fig3.update_layout(title='<b>Marks Distribution</b>',height=420,plot_bgcolor='#f9f9f9')
fig3.show()

In [29]:
# ── Cell 12: Chart 3 – CLO Attainment ────────────────────────────────────────
if CLO_STATS:
    clo_avgs = [CLO_STATS[c]['avg'] for c in CLO_NAMES]
    clo_clrs = [C['success'] if v>=CLO_THRESHOLD else C['danger'] for v in clo_avgs]

    fig5 = go.Figure(go.Bar(x=CLO_NAMES,y=clo_avgs,marker_color=clo_clrs,
        text=[f'{v:.1f}%' for v in clo_avgs],textposition='outside'))
    fig5.add_hline(y=CLO_THRESHOLD,line_dash='dash',line_color=C['danger'],
        annotation_text=f'Threshold {CLO_THRESHOLD}%',annotation_font_color=C['danger'])
    fig5.update_layout(title=f'<b>CLO Attainment vs Threshold ({CLO_THRESHOLD}%)</b>',
        height=420,yaxis=dict(range=[0,110],gridcolor='#eee'),plot_bgcolor='#f9f9f9')
    fig5.show()

    # CLO pass/fail grouped
    fig6 = go.Figure()
    fig6.add_trace(go.Bar(name='Pass',x=CLO_NAMES,
        y=[CLO_STATS[c]['pass_cnt'] for c in CLO_NAMES],marker_color=C['success'],
        text=[CLO_STATS[c]['pass_cnt'] for c in CLO_NAMES],textposition='inside'))
    fig6.add_trace(go.Bar(name='Fail',x=CLO_NAMES,
        y=[CLO_STATS[c]['fail_cnt'] for c in CLO_NAMES],marker_color=C['danger'],
        text=[CLO_STATS[c]['fail_cnt'] for c in CLO_NAMES],textposition='inside'))
    fig6.update_layout(barmode='group',title='<b>CLO Pass/Fail Count</b>',height=420,
        plot_bgcolor='#f9f9f9',legend=dict(orientation='h',y=1.1))
    fig6.show()

    # Radar
    if len(CLO_NAMES)>=2:
        cats=CLO_NAMES+[CLO_NAMES[0]]; atts=clo_avgs+[clo_avgs[0]]
        fig7=go.Figure()
        fig7.add_trace(go.Scatterpolar(r=atts,theta=cats,fill='toself',name='Avg Attainment',
            line_color=C['primary'],fillcolor='rgba(26,115,232,0.22)'))
        fig7.add_trace(go.Scatterpolar(r=[CLO_THRESHOLD]*len(cats),theta=cats,fill='toself',
            name=f'Threshold ({CLO_THRESHOLD}%)',line_color=C['danger'],line_dash='dash',
            fillcolor='rgba(234,67,53,0.07)'))
        fig7.update_layout(polar=dict(radialaxis=dict(visible=True,range=[0,100])),
            title='<b>CLO Radar</b>',height=460,legend=dict(orientation='h',y=-0.1))
        fig7.show()

    # Heatmap
    heat_cols=[c for c in CLO_NAMES if c in df.columns]
    if heat_cols:
        heat=df[['Name']+heat_cols].set_index('Name')
        fig8=go.Figure(go.Heatmap(
            z=heat.values.tolist(),x=heat_cols,y=heat.index.tolist(),
            colorscale=[[0,'#b71c1c'],[CLO_THRESHOLD/100,'#fdd835'],[1,'#1b5e20']],
            zmin=0,zmax=100,text=[[f'{v:.1f}%' for v in r] for r in heat.values.tolist()],
            texttemplate='%{text}',colorbar=dict(title='Attainment %',ticksuffix='%')))
        fig8.update_layout(title='<b>CLO Heatmap (per Student)</b>',
            height=max(420,20*len(heat)),yaxis=dict(tickfont=dict(size=9)),
            margin=dict(l=170,r=40))
        fig8.show()

In [30]:
# ── Cell 13: Chart 4 – Top/Bottom + Box Plot + Mid vs End ────────────────────
n_show=min(TOP_N,max(1,N//2))
_sorted=df.sort_values('Percentage',ascending=False).reset_index(drop=True)
top_df=_sorted.head(n_show); bot_df=_sorted.tail(n_show).sort_values('Percentage')

fig9=make_subplots(1,2,subplot_titles=[f'🏆 Top {n_show}',f'⚠️ Bottom {n_show}'])
fig9.add_trace(go.Bar(x=top_df['Percentage'],y=top_df['Name'],orientation='h',
    marker_color=[gc(g) for g in top_df['Grade']],
    text=[f'{g}|{p:.1f}%' for g,p in zip(top_df['Grade'],top_df['Percentage'])],
    textposition='inside'),1,1)
fig9.add_trace(go.Bar(x=bot_df['Percentage'],y=bot_df['Name'],orientation='h',
    marker_color=C['danger'],
    text=[f'{g}|{p:.1f}%' for g,p in zip(bot_df['Grade'],bot_df['Percentage'])],
    textposition='inside'),1,2)
fig9.update_layout(height=max(380,n_show*35+80),showlegend=False,plot_bgcolor='#f9f9f9')
fig9.update_xaxes(range=[0,105])
fig9.show()

# Box plot
fig10=go.Figure()
for g in _grades_present:
    sub=df[df['Grade']==g]['Percentage']
    if not sub.empty:
        fig10.add_trace(go.Box(y=sub,name=g,marker_color=gc(g),
                               boxpoints='all',jitter=0.4,pointpos=-1.6))
fig10.update_layout(title='<b>Marks Spread by Grade</b>',height=450,
                    showlegend=False,plot_bgcolor='#f9f9f9')
fig10.show()

# Mid vs End scatter (Theory)
if 'Mid' in df.columns and 'End' in df.columns and df['Mid'].sum()>0:
    fig11=px.scatter(df,x='Mid',y='End',color='Grade',size='Percentage',size_max=22,
        hover_data={'Name':True,'Reg_No':True,'Percentage':':.2f'},
        title='<b>Mid Term vs End Term Scatter</b>')
    fig11.update_layout(height=460,plot_bgcolor='#f9f9f9')
    fig11.show()

In [31]:
# ── Cell 14: Student data table ───────────────────────────────────────────────
pd.set_option('display.max_rows',100)
base_cols=['Sr','Reg_No','Name','TotalMarks','Grade','Pass_Fail']+CLO_NAMES
disp=[c for c in base_cols if c in df.columns]
print('\n📊 STUDENT REPORT')
display(df[disp].style
    .background_gradient(subset=['TotalMarks'],cmap='RdYlGn',vmin=0,vmax=100)
    .applymap(lambda v:'background-color:#c8e6c9;color:#1b5e20' if v=='Pass'
              else 'background-color:#ffcdd2;color:#b71c1c' if v=='Fail' else '',
              subset=['Pass_Fail'])
    .format({c:'{:.2f}' for c in df[disp].select_dtypes('float').columns})
    .set_properties(**{'font-size':'11px'}))

if CLO_STATS:
    print('\n📋 CLO ATTAINMENT')
    display(pd.DataFrame([{'CLO':c,'Avg Attainment':f"{CLO_STATS[c]['avg']:.1f}%",
        'Pass':CLO_STATS[c]['pass_cnt'],'Fail':CLO_STATS[c]['fail_cnt'],
        'Pass%':f"{CLO_STATS[c]['pass_pct']:.1f}%",
        'Status':'✅ Met' if CLO_STATS[c]['avg']>=CLO_THRESHOLD else '❌ Not Met'}
        for c in CLO_NAMES]))


📊 STUDENT REPORT


,Sr,Reg_No,Name,TotalMarks,Grade,Pass_Fail,CLO 1,CLO 2,CLO 3,CLO 4
0,1,052-25-126005,Naveed Ul Hassan,3.67,F,Fail,0.00,30.00,3.33,0.00
1,2,052-25-126006,Muhammad Zaid,38.50,F,Fail,23.81,15.00,51.53,45.00
2,3,052-25-126020,Muhammad Sudais,44.33,F,Fail,19.05,56.25,68.75,45.00
3,4,052-25-126028,Hina Qamar,85.67,A-,Pass,93.33,92.50,80.08,92.31
4,5,052-25-126035,Muhammad Talha,40.00,F,Fail,29.05,42.50,38.91,56.54
5,6,052-25-126036,Yahya Sohail Khan,61.92,C,Pass,34.29,62.50,54.23,91.15
6,7,052-25-126045,Yasir Abbas,62.17,C,Pass,37.62,62.50,66.07,68.08
7,8,052-25-126046,Numan Khan,64.83,C,Pass,63.33,68.75,65.14,64.23
8,9,052-25-126058,Zainab Hanif,90.00,A,Pass,90.48,95.00,92.50,84.62
9,10,052-25-126072,Aryan Khalid,77.92,B,Pass,75.24,71.25,65.52,91.15



📋 CLO ATTAINMENT


,CLO,Avg Attainment,Pass,Fail,Pass%,Status
0,CLO 1,43.3%,10,13,43.5%,❌ Not Met
1,CLO 2,46.8%,11,12,47.8%,❌ Not Met
2,CLO 3,50.5%,14,9,60.9%,✅ Met
3,CLO 4,52.9%,13,10,56.5%,✅ Met


In [32]:
# ── Cell 15: Export HTML Dashboard & CSV ──────────────────────────────────────
import re as _re
_safe = _re.sub(r'[^\w]','_',COURSE_CODE)

# Build charts list for HTML
_all_figs=[('Pass/Fail & Grade Pies',fig1),('Grade Distribution',fig2),
           ('Marks Distribution',fig3)]
if CLO_STATS:
    _all_figs+=[("CLO Attainment",fig5),("CLO Pass/Fail",fig6)]
    if len(CLO_NAMES)>=2: _all_figs.append(("CLO Radar",fig7))
    if heat_cols: _all_figs.append(("CLO Heatmap",fig8))
_all_figs+=[("Top & Bottom",fig9),("Box Plot",fig10)]

_charts=''.join(
    f'<div class="cw"><h3>{name}</h3>'+
    fig.to_html(full_html=False,include_plotlyjs="cdn" if i==0 else False)+"</div>"
    for i,(name,fig) in enumerate(_all_figs))

_html=f"""<!DOCTYPE html><html><head><meta charset="UTF-8">
<title>{COURSE_CODE} Dashboard</title>
<style>body{{font-family:'Segoe UI',sans-serif;background:#f4f6f9;margin:0;padding:16px}}
.banner{{background:linear-gradient(135deg,#1a237e,#1565c0);color:#fff;
         border-radius:12px;padding:18px 26px;margin-bottom:20px}}
.banner h1{{margin:0 0 5px;font-size:22px}}.banner p{{margin:0;font-size:12px;opacity:.85}}
.cw{{background:#fff;border-radius:10px;padding:14px;margin-bottom:18px;
     box-shadow:0 2px 8px rgba(0,0,0,.08)}}
.cw h3{{font-size:14px;color:#1a237e;margin:0 0 10px}}</style></head><body>
<div class="banner"><h1>🎓 {COURSE_CODE} — {COURSE_TITLE}</h1>
<p>Faculty: {FACULTY} | {PROGRAM} | {BATCH} | {SEMESTER}</p></div>
{_charts}</body></html>"""

_html_file=f'{_safe}_Dashboard.html'
with open(_html_file,'w',encoding='utf-8') as f: f.write(_html)
print(f'✅ HTML dashboard: {_html_file}')

_csv_file=f'{_safe}_Report.csv'
df[disp].to_csv(_csv_file,index=False,float_format='%.2f')
print(f'✅ CSV report: {_csv_file}')

if IN_COLAB:
    from google.colab import files as _gf2
    _gf2.download(_html_file)
    _gf2.download(_csv_file)
    print('📥 Downloads started')

print('\n🎉 All done!')

✅ HTML dashboard: CMC_112_Dashboard.html
✅ CSV report: CMC_112_Report.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloads started

🎉 All done!
